<a href="https://colab.research.google.com/github/pintoliveira/DSAiVE/blob/main/LLM_Wiki_Ingestion_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Wiki Ingestion Pipeline

Reproduces the Codex/Antigravity LLM Wiki workflow using Gemini API calls.

```
raw document -> extract text -> call LLM API -> create/update wiki pages -> update index.md -> append log.md
```

**Modes:**
- **Option A (Flat):** All pages written to `wiki/`
- **Option B (5-C Taxonomy):** Pages routed to `wiki/knowledge/<category>/`

Run all cells in order. To ingest additional documents, change `raw_filename` in Cell 9 and re-run only that cell.

## Cell 1 — Environment Setup

In [ ]:
# @title Cell 1 — Environment Setup

# Detect Colab and configure environment
from google.colab import auth
auth.authenticate_user()

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, userdata
    import os

    drive.mount('/content/drive')
    !pip install -q google-genai python-docx pandas pymupdf beautifulsoup4

    os.environ["GEMINI_API_KEY"] = userdata.get("GOOGLE_API")
    print("\u2705 Colab environment configured.")
else:
    print("\u26a0\ufe0f Not running in Colab. Configure paths manually.")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 57.1 MB/s eta 0:00:00
✅ Colab environment configured.


In [ ]:
# @title Cell 1.2 — Environment Setup

import google.auth
from google.cloud import aiplatform

# Define your Google Cloud Project ID
PROJECT_ID = userdata.get('PROJECT_ID')
LOCATION = "global" # or your preferred region

# Initialize Vertex AI with your project details
aiplatform.init(project=PROJECT_ID, location=LOCATION)

# Now you can use Vertex AI SDKs securely
print("Authenticated successfully to GCP project:", PROJECT_ID)

Authenticated successfully to GCP project: upkjo-498315


In [ ]:
# @title Cell 1.3 — Environment Setup

from google import genai

# 2. Initialize the client to use Vertex AI
client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION
)

# 3. Request content generation from Gemini 3.5 Flash
try:
    response = client.models.generate_content(
        model='gemini-3.5-flash',
        contents='Write a short, professional welcome message for a developer starting to use Gemini 3.5 Flash.'
    )
    print("✅ Successfully called Gemini 3.5 Flash!")
    print("\nResponse:")
    print(response.text)

except Exception as e:
    print(f"❌ Failed to call the model. Error details:")
    print(e)

✅ Successfully called Gemini 3.5 Flash!

Response:
**Subject: Welcome to Gemini 3.5 Flash!**

Welcome to the Gemini developer community! 

We are excited to see what you build with Gemini 3.5 Flash. Engineered for speed and efficiency, this model is designed to deliver high-frequency, low-latency performance across multimodal tasks without compromising on quality. 

To help you get started quickly:
* **Get your API Key:** Head over to Google AI Studio to set up your credentials.
* **Explore the Docs:** Check out our [Quickstart Guide] and [API Reference] to integrate the model into your stack.
* **Join the Community:** Connect with fellow developers in our forums to share ideas and get support.

Happy coding, and welcome aboard!

Best regards,

**The Gemini Developer Team**


## Cell 2 — Path Resolution & Instructions

In [ ]:
# @title Cell 2.1 — Path Resolution & Instructions

from pathlib import Path
try:
    IN_COLAB
except NameError:
    IN_COLAB = False

if IN_COLAB:
    DATA_DIR = Path("/content/drive/Shareddrives/Python/DSAiVE/10_upk_antigravity/data")
else:
    DATA_DIR = Path(r"G:/Shared drives/Python/DSAiVE/10_upk_antigravity/data")

RAW_DIR = DATA_DIR / "raw"
WIKI_DIR = DATA_DIR / "wiki"
LOCAL_BRAINS_DIR = DATA_DIR / "local_brains" # <-- ADDED

# Ensure directory structures exist
LOCAL_BRAINS_DIR.mkdir(parents=True, exist_ok=True)
WIKI_DIR.mkdir(parents=True, exist_ok=True)

instructions = (DATA_DIR / "AGENTS.md").read_text(encoding="utf-8")
print(f"✅ DATA_DIR = {DATA_DIR}")
print(f"   Instructions loaded: {len(instructions)} chars")
print(f"   Local Brains directory initialized at: {LOCAL_BRAINS_DIR}")

✅ DATA_DIR = /content/drive/Shareddrives/Python/DSAiVE/10_upk_antigravity/data
   Instructions loaded: 1844 chars
   Local Brains directory initialized at: /content/drive/Shareddrives/Python/DSAiVE/10_upk_antigravity/data/local_brains


In [ ]:
# @title Cell 2.2 — Path Resolution & Instructions

# 🔍 QUICK ACCESS TEST
from pathlib import Path

# 1. Verify DATA_DIR definition
try:
    print(f"📂 DATA_DIR path: {DATA_DIR}")
    print(f"🧠 LOCAL_BRAINS_DIR path: {LOCAL_BRAINS_DIR}")
except NameError:
    print("❌ Error: Run Cell 2 first to define your paths.")

# 2. Check if the directory physically exists in your Colab runtime
if LOCAL_BRAINS_DIR.exists():
    print("✅ LOCAL_BRAINS_DIR is accessible.")

    # List all subfolders (Conversation IDs)
    brain_folders = [d for d in LOCAL_BRAINS_DIR.iterdir() if d.is_dir()]
    print(f"📂 Found {len(brain_folders)} local brain conversation folders.")

    for idx, folder in enumerate(brain_folders[:5]): # Show first 5
        print(f"   └── [{idx}] Folder: {folder.name}")
        transcript_path = folder / "transcript.jsonl"
        if transcript_path.exists():
            size_kb = transcript_path.stat().st_size / 1024
            print(f"       ✅ transcript.jsonl found ({size_kb:.2f} KB)")
        else:
            print("       ❌ No transcript.jsonl found in this folder yet.")

    if len(brain_folders) == 0:
        print("ℹ️ Path exists, but no folders found. Ensure your local sync script is running on your PC.")
else:
    print("❌ LOCAL_BRAINS_DIR does not exist. Check if your Google Drive is mounted (Cell 1) or path is correct (Cell 2).")

📂 DATA_DIR path: /content/drive/Shareddrives/Python/DSAiVE/10_upk_antigravity/data
🧠 LOCAL_BRAINS_DIR path: /content/drive/Shareddrives/Python/DSAiVE/10_upk_antigravity/data/local_brains
✅ LOCAL_BRAINS_DIR is accessible.
📂 Found 0 local brain conversation folders.
ℹ️ Path exists, but no folders found. Ensure your local sync script is running on your PC.


## Cell 3 — Text Extractors

In [ ]:
# @title  Cell 3 — Text Extractors

def extract_docx(path: Path) -> str:
    from docx import Document
    doc = Document(str(path))
    return "\n".join(p.text for p in doc.paragraphs if p.text.strip())

def extract_pdf(path: Path) -> str:
    import fitz  # PyMuPDF
    doc = fitz.open(str(path))
    return "\n".join(page.get_text() for page in doc)

def extract_txt(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="replace")

def extract_csv_preview(path: Path, rows: int = 50) -> str:
    import pandas as pd
    df = pd.read_csv(path)
    return df.head(rows).to_markdown(index=False)

def extract_html(path: Path) -> str:
    from bs4 import BeautifulSoup
    html_content = path.read_text(encoding="utf-8", errors="replace")
    soup = BeautifulSoup(html_content, "html.parser")
    for s in soup(["script", "style"]):
        s.decompose()
    text = soup.get_text(separator="\n")
    lines = (line.strip() for line in text.splitlines())
    return "\n".join(line for line in lines if line)

# ─── ADDED JSONL EXTRACTOR FOR BRAIN LOGS ───
def extract_jsonl_transcript(path: Path) -> str:
    import json
    lines = []
    try:
        with open(path, "r", encoding="utf-8") as f:
            for idx, line in enumerate(f):
                if not line.strip():
                    continue
                data = json.loads(line)

                # Normalize typical Antigravity log schemas (role/content or type/parts)
                role = str(data.get("role", data.get("type", "unknown"))).upper()

                content = ""
                if "parts" in data:
                    parts = data["parts"]
                    if isinstance(parts, list):
                        content = " ".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in parts)
                    else:
                        content = str(parts)
                elif "content" in data:
                    content = str(data["content"])
                elif "text" in data:
                    content = str(data["text"])
                else:
                    content = json.dumps(data)

                lines.append(f"### [{idx}] {role}\n{content.strip()}")
    except Exception as e:
        return f"Error reading transcript JSONL: {e}"
    return "\n\n".join(lines)
# ───────────────────────────────────────────

def extract_raw_text(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix == ".docx":
        return extract_docx(path)
    if suffix == ".pdf":
        return extract_pdf(path)
    if suffix in [".txt", ".md"]:
        return extract_txt(path)
    if suffix == ".csv":
        return extract_csv_preview(path)
    if suffix in [".html", ".htm"]:
        return extract_html(path)
    # ─── ADDED ROUTE FOR LOG TRANSCRIPTS ───
    if suffix == ".jsonl":
        return extract_jsonl_transcript(path)
    # ────────────────────────────────────────
    raise ValueError(f"Unsupported file type: {suffix}")

print("✅ Extractors (including HTML and JSONL) defined.")

✅ Extractors (including HTML and JSONL) defined.


## Cell 4 — Wiki Context Reader

In [ ]:
# @title Cell 4 — Wiki Context Reader

import json

def read_existing_wiki_context(wiki_dir: Path, taxonomy_mode: bool = False) -> dict:
    wiki_dir.mkdir(parents=True, exist_ok=True)
    if taxonomy_mode:
        pages = []
        for cat in ["concepts", "code", "clarifications", "claims", "commons"]:
            cat_dir = wiki_dir / "knowledge" / cat
            if cat_dir.exists():
                pages.extend(sorted(cat_dir.glob("*.md")))
    else:
        pages = sorted(wiki_dir.glob("*.md"))

    # ─── ADDED MANIFEST READER ───
    manifest_path = wiki_dir / "manifest.json"
    manifest_data = {}
    if manifest_path.exists():
        try:
            manifest_data = json.loads(manifest_path.read_text(encoding="utf-8"))
        except Exception as e:
            print(f"⚠️ Warning: Could not parse existing manifest.json: {e}")
    # ─────────────────────────────

    return {
        "files": [p.name for p in pages],
        "index": (wiki_dir / "index.md").read_text(encoding="utf-8") if (wiki_dir / "index.md").exists() else "",
        "log": (wiki_dir / "log.md").read_text(encoding="utf-8") if (wiki_dir / "log.md").exists() else "",
        "manifest": manifest_data  # <-- ADDED
    }
print("✅ Wiki context reader configured to load manifest.json.")

✅ Wiki context reader configured to load manifest.json.


## Cell 5 — Prompt Builder

In [ ]:
# @title Cell 5 — Prompt Builder

from datetime import date
import json

def build_ingest_prompt(instructions: str, raw_filename: str, raw_text: str, wiki_context: dict) -> str:
    today = date.today().isoformat()
    return f"""
You are maintaining a local markdown LLM Wiki.

Follow these instructions:
{instructions}

Today is {today}.

Raw source filename:
raw/{raw_filename}

Existing wiki files:
{json.dumps(wiki_context["files"], ensure_ascii=False, indent=2)}

Existing wiki/index.md:
{wiki_context["index"]}

Existing wiki/log.md:
{wiki_context["log"][-4000:]}

Raw source text:
<<<RAW_TEXT
{raw_text}
RAW_TEXT

Task:
Create or update focused atomic markdown wiki pages.
Every page must start with YAML frontmatter:
---
title: "Page Title"
tags: [concept, entity, project]
updated: {today}
---

Use [[Wiki Link]] syntax for internal links.
Keep pages below 400 lines where possible.
Do not modify the raw source.

Return only valid JSON with this schema:
{{{{
  "pages": [
    {{{{
      "filename": "Page Title.md",
      "category": "Concepts",
      "content": "complete markdown content"
    }}}}
  ],
  "index_md": "complete updated index.md content",
  "log_entry": "one markdown bullet or section describing the update"
}}}}
"""

print("\u2705 Prompt builder defined.")

✅ Prompt builder defined.


## Cell 6 — Gemini API Client

In [ ]:
# @title Cell 6.1 — GCP API Client

import json

def call_gemini_json(prompt: str, model: str = "gemini-3.5-flash") -> dict:
    schema = {
        "type": "OBJECT",
        "properties": {
            "pages": {
                "type": "ARRAY",
                "items": {
                    "type": "OBJECT",
                    "properties": {
                        "filename": {"type": "STRING"},
                        "category": {"type": "STRING"},
                        "content": {"type": "STRING"}
                    },
                    "required": ["filename", "category", "content"]
                }
            },
            "index_md": {"type": "STRING"},
            "log_entry": {"type": "STRING"},
            "manifest": {
                "type": "OBJECT",
                "properties": {
                    "wiki_metadata": {
                        "type": "OBJECT",
                        "properties": {
                            "name": {"type": "STRING"},
                            "updated": {"type": "STRING"},
                            "version": {"type": "STRING"}
                        }
                    },
                    "index_mapping": {
                        "type": "OBJECT",
                        "properties": {
                            "concepts": {"type": "OBJECT"},
                            "code": {"type": "OBJECT"},
                            "clarifications": {"type": "OBJECT"},
                            "claims": {"type": "OBJECT"},
                            "commons": {"type": "OBJECT"}
                        }
                    },
                    "pages_registry": {
                        "type": "ARRAY",
                        "items": {"type": "OBJECT"}
                    }
                },
                "required": ["wiki_metadata", "index_mapping", "pages_registry"]
            }
        },
        "required": ["pages", "index_md", "log_entry", "manifest"]
    }

    # Executing the call via Vertex AI with structured JSON outputs
    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": schema,
            "temperature": 0.0,
        },
    )

    text = response.text
    for _ in range(50):
        try:
            return json.loads(text)
        except json.JSONDecodeError as e:
            if "Invalid \\escape" in str(e):
                pos = e.pos
                idx = text.rfind('\\', 0, pos + 1)
                if idx != -1:
                    text = text[:idx] + '\\\\' + text[idx+1:]
                    continue
            raise e

print("✅ Vertex AI Client initialized successfully.")

✅ Vertex AI Client initialized successfully.


In [ ]:
# @title Cell 6.2 — Gemini API Client

import os
from google import genai

# Double check that your key is actually being read by your script
if not os.environ.get("GEMINI_API_KEY"):
     print("Warning: GEMINI_API_KEY environment variable is not set!")

# Initialize the modern client
client = genai.Client()

try:
     response = client.models.generate_content(
         model="gemini-2.5-flash",
         contents="Say 'Hello World' if you can hear me!"
     )
     print("Success! Gemini says:")
     print(response.text)
except Exception as e:
     print(f"An error occurred: {e}")

An error occurred: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API Key not found. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API Key not found. Please pass a valid API key.'}]}}


## Cell 7 — File Writers

In [ ]:
import hashlib
import json
from pathlib import Path
from datetime import date

def safe_wiki_path(wiki_dir: Path, filename: str) -> Path:
    if "/" in filename or "\\" in filename:
        raise ValueError(f"Nested paths are not allowed: {filename}")
    if not filename.endswith(".md"):
        raise ValueError(f"Wiki page must be markdown: {filename}")
    return wiki_dir / filename

def write_wiki_update(wiki_dir: Path, result: dict, taxonomy_mode: bool = False):
    wiki_dir.mkdir(parents=True, exist_ok=True)

    for page in result["pages"]:
        filename = page["filename"]
        content = page["content"]
        category = page.get("category", "").strip().lower()

        if taxonomy_mode and category in ["concepts", "code", "clarifications", "claims", "commons"]:
            target_dir = wiki_dir / "knowledge" / category
        else:
            target_dir = wiki_dir

        target_dir.mkdir(parents=True, exist_ok=True)
        path = safe_wiki_path(target_dir, filename)
        path.write_text(content, encoding="utf-8")

    (wiki_dir / "index.md").write_text(result["index_md"], encoding="utf-8")

    log_path = wiki_dir / "log.md"
    existing_log = log_path.read_text(encoding="utf-8") if log_path.exists() else ""
    new_log = existing_log.rstrip() + "\n\n" + result["log_entry"].rstrip() + "\n"
    log_path.write_text(new_log, encoding="utf-8")

# Manifest & History management helpers
def calculate_file_hash(file_path: Path) -> str:
    """Calculate SHA-256 hash of a file to accurately detect modifications."""
    sha256 = hashlib.sha256()
    with open(file_path, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256.update(byte_block)
    return sha256.hexdigest()

def load_history(history_path: Path) -> dict:
    """Load ingestion history file."""
    if history_path.exists():
        try:
            with open(history_path, "r", encoding="utf-8") as f:
                return json.load(f)
        except json.JSONDecodeError:
            return {}
    return {}

def save_history(history_path: Path, history_data: dict):
    """Save ingestion history file."""
    history_path.parent.mkdir(parents=True, exist_ok=True)
    with open(history_path, "w", encoding="utf-8") as f:
        json.dump(history_data, f, indent=4, ensure_ascii=False)

def load_manifest() -> dict:
    manifest_path = WIKI_DIR / "manifest.json"
    if manifest_path.exists():
        try:
            return json.loads(manifest_path.read_text(encoding="utf-8"))
        except Exception as e:
            print(f"⚠️ Warning: Could not parse manifest.json. Reinitializing. Error: {e}")
    return {"pages_registry": [], "ingested_files": {}}

def save_manifest(manifest: dict):
    manifest_path = WIKI_DIR / "manifest.json"
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

def update_manifest_state(manifest: dict, result_pages: list, source_filename: str = None, thread_id: str = None, next_step: int = 0):
    registry = manifest.setdefault("pages_registry", [])
    for page in result_pages:
        filename = page["filename"]
        category = page.get("category", "Concepts")

        existing_entry = None
        for entry in registry:
            if entry["filename"] == filename:
                existing_entry = entry
                break

        if not existing_entry:
            existing_entry = {
                "filename": filename,
                "category": category,
                "verified": False,
                "source_grounding": {
                    "files": [],
                    "antigravity_threads": []
                }
            }
            registry.append(existing_entry)
        else:
            existing_entry["category"] = category

        grounding = existing_entry.setdefault("source_grounding", {})
        if source_filename:
            files_list = grounding.setdefault("files", [])
            if source_filename not in files_list:
                files_list.append(source_filename)

    if source_filename and not thread_id:
        manifest.setdefault("ingested_files", {})[source_filename] = {
            "ingested_at": date.today().isoformat()
        }

print("✅ File writers, manifest, and history helpers defined.")

✅ File writers, manifest, and history helpers defined.


## Cell 8 — Health Check

In [ ]:
# @title Cell 8 — Health Check

import re

def check_wiki_health(wiki_dir: Path, taxonomy_mode: bool = False):
    if taxonomy_mode:
        files = []
        for cat in ["concepts", "code", "clarifications", "claims", "commons"]:
            cat_dir = wiki_dir / "knowledge" / cat
            if cat_dir.exists():
                files.extend(cat_dir.glob("*.md"))
        for root_file in ["index.md", "log.md"]:
            if (wiki_dir / root_file).exists():
                files.append(wiki_dir / root_file)
    else:
        files = list(wiki_dir.glob("*.md"))

    stems = {p.stem for p in files}
    title_to_stem = {}
    missing_frontmatter = []
    broken_links = []
    incoming = {p.stem: 0 for p in files}

    for p in files:
        text = p.read_text(encoding="utf-8")
        if not text.startswith("---\n"):
            missing_frontmatter.append(p.name)
        m = re.search(r'^title: "([^"]+)"', text, re.M)
        if m:
            title_to_stem[m.group(1)] = p.stem

    for p in files:
        text = p.read_text(encoding="utf-8")
        for link in re.findall(r"\[\[([^\]|#]+)", text):
            link = link.strip()
            target = link if link in stems else title_to_stem.get(link)
            if target:
                incoming[target] = incoming.get(target, 0) + 1
            else:
                broken_links.append((p.name, link))

    orphan_pages = [
        stem for stem, count in incoming.items()
        if count == 0 and stem not in {"index", "log"}
    ]

    return {
        "files": len(files),
        "missing_frontmatter": missing_frontmatter,
        "broken_links": broken_links,
        "orphan_pages": orphan_pages,
    }

print("\u2705 Health check defined.")

✅ Health check defined.


## Cell 9 — 🚀 Run Ingestion

**Edit `raw_filename` and `TAXONOMY_MODE` below, then run this cell.**

To ingest additional documents later, change `raw_filename` and re-run only this cell.

In [ ]:
# ──── CONFIGURATION ────
TAXONOMY_MODE = False                  # True = 5-C folders (Option B), False = flat wiki/ (Option A)
FORCE_REPROCESS = False                # Force reprocessing of files (ignore hashing/history)
PROCESS_MODE = "reprocess_if_changed"  # Ingestion cache mode: "reprocess_if_changed", "skip_if_processed", "always"

# Set either raw_filename (for single file) OR raw_dir (for directory ingestion)
raw_filename = "Vertex ai hist 042026.docx"  # Set to name of single file under raw/, or None
raw_dir = None                               # Set to directory path, or None
# ───────────────────────

# Path to the specific exported conversation file:
raw_filename = "/content/drive/Shareddrives/Python/DSAiVE/10_upk_antigravity/Exported_Conversations/0bfdeded-b23b-4be4-af6b-d85dc77253bf.md"
raw_dir = None
# ───────────────────────

# Batch process all files in the folder:
raw_filename = None
raw_dir = "/content/drive/Shareddrives/Python/DSAiVE/10_upk_antigravity/Exported_Conversations"
# ───────────────────────

import os
from datetime import datetime
from pathlib import Path

MANIFEST_PATH = WIKI_DIR / "manifest.json"
HISTORY_FILE = DATA_DIR / "processed" / "ingestion_history.json"

def process_single_raw_file(raw_path: Path, instructions: str, manifest: dict) -> bool:
    source_filename = raw_path.name
    source_label = f"raw/{source_filename}"

    # Check processing status using ingestion_history.json
    process_mode = "always" if FORCE_REPROCESS else PROCESS_MODE
    history = load_history(HISTORY_FILE)
    current_hash = calculate_file_hash(raw_path)
    file_record = history.get(source_filename)

    skip_processing = False
    if file_record:
        last_processed_hash = file_record.get("hash")
        last_processed_time = file_record.get("last_processed")
        print(f"[History] Found record: '{source_filename}' was previously processed on {last_processed_time}")

        if process_mode == "skip_if_processed":
            print(f"[History] Skipping: File has been processed once before (mode: '{process_mode}').")
            skip_processing = True
        elif process_mode == "reprocess_if_changed":
            if current_hash == last_processed_hash:
                print(f"[History] Skipping: File content is unchanged (SHA-256 hash matches).")
                skip_processing = True
            else:
                print(f"[History] Modified: File content changes detected. Proceeding to re-process...")
    else:
        print(f"[History] New File: '{source_filename}' has not been processed yet. Processing...")

    if skip_processing:
        return False

    print(f"[File] Reading document: {raw_path.name}")
    try:
        raw_text = extract_raw_text(raw_path)
    except Exception as e:
        print(f"❌ Error extracting text from {raw_path.name}: {e}")
        return False

    print(f"[Text] Extracted {len(raw_text)} characters.")

    # Read fresh wiki context
    wiki_context = read_existing_wiki_context(WIKI_DIR, taxonomy_mode=TAXONOMY_MODE)
    print(f"[Wiki] Loaded existing wiki context ({len(wiki_context['files'])} pages).")

    # Build Prompt
    prompt = build_ingest_prompt(instructions, source_filename, raw_text, wiki_context)

    # Generate JSON Updates via API
    print("[API] Calling Gemini API...")
    result = call_gemini_json(prompt)
    print(f"[Success] Gemini returned updates for {len(result['pages'])} wiki pages.")

    # Write changes to files
    write_wiki_update(WIKI_DIR, result, taxonomy_mode=TAXONOMY_MODE)
    print(f"[Save] Saved updates to wiki files under {WIKI_DIR}")

    # Update manifest structure programmatically
    update_manifest_state(
        manifest=manifest,
        result_pages=result["pages"],
        source_filename=source_filename
    )
    save_manifest(manifest)
    print("[Manifest] Updated manifest.json registry.")

    # Update ingestion history for raw files
    history = load_history(HISTORY_FILE)
    history[source_filename] = {
        "hash": current_hash,
        "last_processed": datetime.now().isoformat(),
        "file_size_bytes": raw_path.stat().st_size
    }
    save_history(HISTORY_FILE, history)
    print(f"[History] Logged processing status for '{source_filename}' to ingestion history.")
    return True

# ──── EXECUTION FLOW ────
manifest = load_manifest()

if raw_dir:
    dir_path = Path(raw_dir)
    if not dir_path.exists() or not dir_path.is_dir():
        raise FileNotFoundError(f"[Error] Directory not found: {raw_dir}")

    print(f"[Dir] Scanning directory: {dir_path.resolve()}")
    # Find all files in the directory (excluding subfolders)
    files = sorted([p for p in dir_path.iterdir() if p.is_file() and p.suffix.lower() in [".docx", ".pdf", ".txt", ".md", ".csv", ".html", ".htm", ".jsonl"]])

    if not files:
        print("[Dir] No supported files found in the directory.")
    else:
        print(f"[Dir] Found {len(files)} files to process.")
        processed_count = 0
        for idx, file_path in enumerate(files):
            print(f"\n--- Ingesting [{idx+1}/{len(files)}]: {file_path.name} ---")
            if process_single_raw_file(file_path, instructions, manifest):
                processed_count += 1
        print(f"\n[Status] Directory ingestion complete! Processed/updated {processed_count} files.")

elif raw_filename:
    raw_path = RAW_DIR / raw_filename
    if not raw_path.exists():
        raw_path = Path(raw_filename)
        if not raw_path.exists():
            raise FileNotFoundError(f"[Error] Raw file not found at {RAW_DIR / raw_filename} or {raw_filename}")

    processed = process_single_raw_file(raw_path, instructions, manifest)
    if not processed:
        print("\n[Status] Task completed without modifications (no processing required).")

else:
    print("[Error] Please specify either raw_filename or raw_dir in the configuration.")

# Check health
health = check_wiki_health(WIKI_DIR, taxonomy_mode=TAXONOMY_MODE)
print(f"\n[Health] Wiki Health Report:")
for key, value in health.items():
    print(f"   {key}: {value}")

[Dir] Scanning directory: /content/drive/Shareddrives/Python/DSAiVE/10_upk_antigravity/Exported_Conversations
[Dir] Found 16 files to process.

--- Ingesting [1/16]: 0bfdeded-b23b-4be4-af6b-d85dc77253bf.md ---
[History] Found record: '0bfdeded-b23b-4be4-af6b-d85dc77253bf.md' was previously processed on 2026-06-06T15:23:44.510453
[History] Skipping: File content is unchanged (SHA-256 hash matches).

--- Ingesting [2/16]: 12a77363-dc4b-468e-9c42-0cc6bf6343b2.md ---
[History] New File: '12a77363-dc4b-468e-9c42-0cc6bf6343b2.md' has not been processed yet. Processing...
[File] Reading document: 12a77363-dc4b-468e-9c42-0cc6bf6343b2.md
[Text] Extracted 1133 characters.
[Wiki] Loaded existing wiki context (78 pages).
[API] Calling Gemini API...
[Success] Gemini returned updates for 1 wiki pages.
[Save] Saved updates to wiki files under /content/drive/Shareddrives/Python/DSAiVE/10_upk_antigravity/data/wiki
[Manifest] Updated manifest.json registry.
[History] Logged processing status for '12a773

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}

In [ ]:
import os
path = "/content/drive/Shareddrives/Python/DSAiVE/10_upk_antigravity/data/wiki"
if os.path.exists(path):
    files = os.listdir(path)
    print(f"✅ Wiki path exists. Number of files found: {len(files)}")
    print("First 10 files:", files[:10])
else:
    print("❌ Wiki path does not exist!")

✅ Wiki path exists. Number of files found: 80
First 10 files: ['Google AI Studio and Context Window Recovery.md', 'log.md', 'manifest.json', 'index.md', 'upK Schedule Visualisation System.md', 'Greedy Decoding.md', 'Vertex AI History 042026 Source.md', 'LLM Proposition Validation.md', 'Semantic Entropy.md', 'Logprobs and White-Box Scoring.md']


In [ ]:
# @title Cell 10 — 🔍 Query index

# ──── PROGRAMMATIC SEARCH CELL ────
query_term = "instance space"  # <-- Change this to search for other terms
# ──────────────────────────────────

def search_wiki(wiki_dir, query_term, taxonomy_mode=True):
    import re
    results = []

    # Resolve target files
    files = []
    if taxonomy_mode:
        for cat in ["concepts", "code", "clarifications", "claims", "commons"]:
            cat_dir = wiki_dir / "knowledge" / cat
            if cat_dir.exists():
                files.extend(cat_dir.glob("*.md"))
        # Include root index and log files
        files.extend(wiki_dir.glob("*.md"))
    else:
        files = list(wiki_dir.glob("*.md"))

    for file_path in files:
        if file_path.is_dir():
            continue
        try:
            content = file_path.read_text(encoding="utf-8")
            # Search content and title
            if re.search(query_term, content, re.IGNORECASE) or re.search(query_term, file_path.name, re.IGNORECASE):
                # Count matches
                matches = len(re.findall(query_term, content, re.IGNORECASE))
                results.append({
                    "title": file_path.stem,
                    "matches": matches,
                    "path": str(file_path.relative_to(wiki_dir))
                })
        except Exception as e:
            continue

    # Sort by number of matches
    results = sorted(results, key=lambda x: x['matches'], reverse=True)
    return results

# Run the search
search_results = search_wiki(WIKI_DIR, query_term, TAXONOMY_MODE)
print(f"🔍 Found {len(search_results)} notes containing '{query_term}':\n")
for idx, res in enumerate(search_results, 1):
    print(f"{idx}. [[{res['title']}]] ({res['matches']} matches)")
    print(f"   Location: {res['path']}\n")

🔍 Found 7 notes containing 'instance space':

1. [[ISA Prediction and Footprint Generation]] (7 matches)
   Location: ISA Prediction and Footprint Generation.md

2. [[Instance Space Analysis]] (5 matches)
   Location: Instance Space Analysis.md

3. [[ISA Space Construction Pipeline]] (5 matches)
   Location: ISA Space Construction Pipeline.md

4. [[log]] (4 matches)
   Location: log.md

5. [[index]] (3 matches)
   Location: index.md

6. [[Instance Space Analysis for Algorithm Testing Source]] (3 matches)
   Location: Instance Space Analysis for Algorithm Testing Source.md

7. [[MATILDA and Timetabling Case Study]] (3 matches)
   Location: MATILDA and Timetabling Case Study.md



In [ ]:
response_1 = response

In [ ]:
# @title Cell 11.2 — 🔍 Query the LLM Wiki (Natural Language Q&A)

from datetime import datetime
import re

# ──── EXECUTION ────
# Enter any natural language question here:
USER_QUESTION = "TEL"

response = ask_wiki(USER_QUESTION, WIKI_DIR)
print("\n📝 ANSWER:")
print(response)

# ──── PERSISTENCE WITH INDEXING ────
qa_log_path = WIKI_DIR / "QA_History.md"
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Determine Version Indexing
version_count = 1
if qa_log_path.exists():
    history_content = qa_log_path.read_text(encoding="utf-8")
    # Count occurrences of the specific question to determine the next version
    version_count = history_content.count(f"**Question:** {USER_QUESTION}") + 1

# Create the entry with indexing reference
persistence_entry = f"""
## Q&A Session: {timestamp} (Version: {version_count})
**Question:** {USER_QUESTION}

**Answer:**
{response}

---
"""

# Write or append to the QA_History file
if not qa_log_path.exists():
    header = """---
title: "Q&A History"
tags: [log, qa]
---

# Wiki Q&A Persistence Log
"""
    qa_log_path.write_text(header + persistence_entry, encoding="utf-8")
else:
    with open(qa_log_path, "a", encoding="utf-8") as f:
        f.write(persistence_entry)

print(f"\n💾 Answer persisted (v{version_count}) to: {qa_log_path}")

🔍 Routing query to relevant wiki pages...
📖 Reading files: ['Vertex AI History 042026 Source.md', 'Instance Space Analysis.md', 'Instance Space Analysis for Algorithm Testing Source.md', 'knowledge/concepts/Semantic Drift and LLM Thread Categorised History Report.md']
🧠 Synthesizing answer...

📝 ANSWER:
Based on the provided documentation, there are **no direct references** to Instance Space Analysis (ISA) in your Vertex AI history [[Vertex AI History 042026 Source.md]]. 

However, there are strong **conceptual connections** between the methodologies used in your Vertex AI developmental history/auditing and the framework of Instance Space Analysis (ISA). These connections include:

### 1. Rejecting "On-Average" Metrics in Favor of Localized, Granular Evaluation
*   **Vertex AI History / Auditing:** Standard evaluation metrics (such as ROUGE or BERTScore) are noted as falling short because they only measure generic lexical similarity [[Semantic Drift and LLM Thread Categorised History R

## Cell 10 — Q&A RAG Query Functions

In [ ]:
def get_all_wiki_files(wiki_dir: Path) -> dict:
    """Helper to find all compiled markdown files relative to the wiki root, ignoring index/log."""
    files = {}
    # Sort files alphabetically by relative path to ensure deterministic prompt context order
    for p in sorted(wiki_dir.rglob("*.md"), key=lambda x: str(x.relative_to(wiki_dir)).lower()):
        if p.name in ["index.md", "log.md"]:
            continue
        rel_path = p.relative_to(wiki_dir)
        files[str(rel_path).replace("\\", "/")] = p
    return files

def get_last_qa_sessions(qa_path: Path, count: int = 5) -> str:
    """Extracts the last `count` Q&A sessions from QA_History.md to avoid context window bloat."""
    if not qa_path.exists():
        return ""
    try:
        content = qa_path.read_text(encoding="utf-8")
        sessions = content.split("## Q&A Session:")
        if len(sessions) <= 1:
            return content
        last_sessions = sessions[-count:]
        reconstructed = ["# Recent Q&A History"]
        for s in last_sessions:
            reconstructed.append(f"## Q&A Session:{s.strip()}")
        return "\n\n".join(reconstructed)
    except Exception as e:
        print(f"⚠️ Warning: Could not parse QA_History.md: {e}")
        return ""

def ask_wiki(query: str, wiki_dir: Path) -> str:
    # 1. Read Index and map of files
    index_content = (wiki_dir / "index.md").read_text(encoding="utf-8") if (wiki_dir / "index.md").exists() else "No index found."
    available_files = get_all_wiki_files(wiki_dir)
    file_list_str = "\n".join(f"- {name}" for name in available_files.keys())

    # 2. Ask Gemini to route the question to the correct pages
    routing_prompt = f"""
You are the Librarian of an LLM Wiki.
Your goal is to identify which specific wiki files are needed to answer this user query:
\"{query}\"

Here is the current table of contents/index:
{index_content}

Here is the list of all available files (ordered alphabetically):
{file_list_str}

Select the files that are relevant and necessary to answer the user query:
- If the query asks about the ENTIRE wiki, ALL files, EACH file, EVERY file, the whole index, or how the files interconnect in general, you MUST select ALL available files.
- \"QA_History.md\" contains logs of past questions and answers. Select it ONLY if the query refers to past questions, answers, or the history of Q&A interactions.
- For specific, targeted questions, select up to 4 most relevant files.
You must output a valid JSON object with the key \"selected_files\" containing a list of strings matching the exact file paths listed above.
If no files are relevant, return an empty list.
"""

    routing_schema = {
        "type": "OBJECT",
        "properties": {
            "selected_files": {
                "type": "ARRAY",
                "items": {"type": "STRING"}
            }
        },
        "required": ["selected_files"]
    }

    route_response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=routing_prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": routing_schema,
            "temperature": 0.0,
        },
    )

    try:
        route_data = json.loads(route_response.text)
        selected_files = route_data.get("selected_files", [])
    except Exception as e:
        print(f"⚠️ Warning: Routing parse error: {e}")
        selected_files = list(available_files.keys())[:3]

    if not selected_files:
        return "I couldn\'t find any relevant pages in the wiki to answer this question."

    # 3. Read content
    context_blocks = []
    for file_rel in selected_files:
        file_path = available_files.get(file_rel)
        if file_path and file_path.exists():
            if file_rel == "QA_History.md":
                content = get_last_qa_sessions(file_path, count=5)
            else:
                content = file_path.read_text(encoding="utf-8")
            context_blocks.append(f"--- START FILE: {file_rel} ---\n{content}\n--- END FILE: {file_rel} ---")

    context_text = "\n\n".join(context_blocks)

    # 4. Synthesize
    synthesis_prompt = f"""
You are an expert Q&A engine answering questions based strictly on the provided LLM Wiki pages.

User\'s Question:
\"{query}\"

Relevant Wiki Context:
{context_text}

Answer the user\'s question clearly, thoroughly, and objectively.
- Rely only on the clear facts directly mentioned in the context.
- When referencing concepts or facts, cite the source page in double brackets, like [[{selected_files[0]}]] or similar.
"""

    answer_response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=synthesis_prompt,
        config={
            "temperature": 0.0,
        }
    )
    return answer_response.text

print("✅ Q&A RAG Query functions defined.")

## Cell 11 — 🚀 Ask the Wiki

**Enter your query in the code below and run this cell. It will print the answer and persist the Q&A session to `QA_History.md`.**

To load a query from a file (e.g., a long question formatted in `.md`), set `query_input = "question.md"` pointing to the file path.

In [ ]:
from datetime import datetime

# ──── Q&A CONFIGURATION ────
query_input = "Please tell me about 'Reframing the TSP.md'"
# ───────────────────────────

query_text = query_input
query_label = query_input

# Check if the query is a file path
possible_paths = [
    Path(query_input),
    WIKI_DIR / query_input,
    RAW_DIR / query_input
]
for path in possible_paths:
    if path.exists() and path.is_file():
        try:
            query_text = path.read_text(encoding="utf-8")
            query_label = f"File: {path.name}"
            print(f"📂 Loaded query from file: {path.resolve()}")
            break
        except Exception as e:
            print(f"⚠️ Warning: Could not read query file {path}: {e}")

print(f"🔍 Asking LLM Wiki: \'{query_label}\'")
response = ask_wiki(query_text, WIKI_DIR)
print("\n=== ANSWER ===")
print(response)
print("==============\n")

# Persist Q&A to QA_History.md
qa_log_path = WIKI_DIR / "QA_History.md"
timestamp_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Determine Version Indexing
version_count = 1
if qa_log_path.exists():
    history_content = qa_log_path.read_text(encoding="utf-8")
    version_count = history_content.count(f"**Question:** {query_label}") + 1

persistence_entry = f"\n## Q&A Session: {timestamp_str} (Version: {version_count})\n"
persistence_entry += f"**Question:** {query_label}\n"
if query_label != query_text:
    persistence_entry += f"\nQuery Content:\n```markdown\n{query_text.strip()}\n```\n"

persistence_entry += f"\n**Answer:**\n{response}\n\n---\n"

if not qa_log_path.exists():
    header = """---
title: "Q&A History"
tags: [log, qa]
---

# Wiki Q&A Persistence Log
"""
    qa_log_path.write_text(header + persistence_entry, encoding="utf-8")
else:
    with open(qa_log_path, "a", encoding="utf-8") as f:
        f.write(persistence_entry)

print(f"💾 Answer persisted (v{version_count}) to: {qa_log_path.name}")

🔍 Asking LLM Wiki: 'Please tell me about 'Reframing the TSP.md''


NameError: name 'ask_wiki' is not defined